# Подход 1. Строгий датасет связанных объектов

Этот notebook запускает `41_СФЕРА_датасет_модель_1.sql`.

В результат попадают только объекты недвижимости, для которых подтверждена цепочка:

`договор → заявка → выбранная задача → связь с объектом → характеристики → условия`.

Преимущество подхода — у каждой строки есть договорный контекст. Недостаток — большая часть договоров не имеет заполненной связи с объектами и не попадает в датасет.

## 1. Библиотеки

Следующую ячейку достаточно выполнить один раз в используемом окружении. Если библиотеки уже установлены, её можно пропустить.

In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]"

In [ ]:
import getpass
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 100)

## 2. Путь к проекту

Ячейка сама найдёт корень проекта, если notebook открыт из папки проекта или `notebooks`.

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for folder in (start, *start.parents):
        if (folder / '41_СФЕРА_датасет_модель_1.sql').is_file():
            return folder
    raise FileNotFoundError('Не найден корень проекта с SQL 41')

PROJECT_ROOT = find_project_root(Path.cwd())
SQL_PATH = PROJECT_ROOT / '41_СФЕРА_датасет_модель_1.sql'
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Корень проекта:', PROJECT_ROOT)
print('SQL:', SQL_PATH)

## 3. Подключение к Сфере

Возьми сервер, порт, базу и логин из свойств рабочего подключения DBeaver. Пароль вводится скрыто и не записывается в notebook.

In [ ]:
SPHERE_HOST = ''       # сервер из DBeaver
SPHERE_PORT = 5432     # порт из DBeaver
SPHERE_DATABASE = ''   # база данных из DBeaver
SPHERE_USER = ''       # логин из DBeaver

if not all([SPHERE_HOST, SPHERE_DATABASE, SPHERE_USER]):
    raise ValueError('Заполни SPHERE_HOST, SPHERE_DATABASE и SPHERE_USER')

password = getpass.getpass('Пароль от Сферы: ')
connection_url = (
    f'postgresql+psycopg://{quote_plus(SPHERE_USER)}:'
    f'{quote_plus(password)}@{SPHERE_HOST}:{SPHERE_PORT}/{SPHERE_DATABASE}'
)
engine = create_engine(connection_url, pool_pre_ping=True)

In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

## 4. Запуск SQL 41

Запрос может выполняться дольше тестового запроса подключения. Не закрывай kernel во время выполнения.

In [ ]:
sql = SQL_PATH.read_text(encoding='utf-8')

with engine.connect() as connection:
    strict_df = pd.read_sql_query(text(sql), connection)

print('Строк:', len(strict_df))
print('Колонок:', len(strict_df.columns))
display(strict_df.head(3))

## 5. Проверка зерна и заполненности

Здесь выводятся только количества. Адреса, ИНН и номера договоров не печатаются.

In [ ]:
required_columns = {
    'contract_id', 'task_id', 'object_id', 'characteristics_id',
    'elementary_obj_type', 'insured_sum', 'full_address'
}
missing_columns = sorted(required_columns - set(strict_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных договоров',
        'Уникальных задач',
        'Уникальных объектов',
        'Уникальных пар задача + объект',
        'Строк с target',
        'Строк с адресом',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(strict_df),
        strict_df['contract_id'].nunique(dropna=True),
        strict_df['task_id'].nunique(dropna=True),
        strict_df['object_id'].nunique(dropna=True),
        strict_df[['task_id', 'object_id']].drop_duplicates().shape[0],
        strict_df['insured_sum'].notna().sum(),
        strict_df['full_address'].fillna('').str.strip().ne('').sum(),
        strict_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
duplicate_keys = (
    strict_df.groupby(['task_id', 'object_id'], dropna=False)
    .size()
    .gt(1)
    .sum()
)
print('Повторных ключей задача + объект:', duplicate_keys)
display(strict_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))

## 6. Сохранение результата

CSV сохраняется в локальную папку, которая исключена из Git. Кодировка `utf-8-sig` и разделитель `;` подходят для русского Excel.

In [ ]:
output_path = OUTPUT_DIR / 'датасет_41_строгий.csv'
strict_df.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print('Сохранено:', output_path)

In [ ]:
engine.dispose()
print('Подключение закрыто')